In [0]:
silver_sales = spark.read.format("delta").load("/Volumes/retail_project/retail_db/retail_files/silver/")
display(silver_sales)

DAILY REVENUE
“How much revenue did we generate per day?”


In [0]:
from pyspark.sql.functions import to_date, sum

gold_daily_revenue = (
    silver_sales
    .withColumn("sales_date", to_date("invoice_date"))
    .groupBy("sales_date")
    .agg(
        sum("total_amount").alias("daily_revenue")
    )
)
display(gold_daily_revenue)

In [0]:
gold_daily_revenue.write.format("delta") \
    .mode("overwrite") \
    .partitionBy("sales_date") \
    .save("/Volumes/retail_project/retail_db/retail_files/gold/")

TOP SELLING PRODUCTS (BUSINESS / CATEGORY TEAM)
“Which products are selling the most?”

In [0]:
from pyspark.sql.functions import sum, desc

gold_top_products = (
    silver_sales
    .groupBy("product_id", "Description")
    .agg(
        sum("quantity").alias("units_sold"),
        sum("total_amount").alias("revenue")
    )
    .orderBy(desc("revenue"))
)
display(gold_top_products)


In [0]:

gold_top_products.write \
    .format("delta") \
    .mode("overwrite") \
    .save("/Volumes/retail_project/retail_db/retail_files/gold_top_products/")

MVC 
“Who are our most valuable customers?”

In [0]:
gold_customer_ltv = (
    silver_sales
    .groupBy("customer_id")
    .agg(
        sum("total_amount").alias("lifetime_value"),
        sum("quantity").alias("total_items")
    )
).display()

In [0]:
gold_customer_ltv.write.format("delta") \
    .mode("overwrite") \
    .save("/Volumes/retail_project/retail_db/retail_files/gold_customer_ltv/")


MONTHLY SALES TREND

In [0]:
from pyspark.sql.functions import date_format

gold_monthly_sales = (
    silver_sales
    .withColumn("year_month", date_format("invoice_date", "yyyy-MM"))
    .groupBy("year_month")
    .agg(
        sum("total_amount").alias("monthly_revenue")
    )
    .orderBy("year_month")
)


In [0]:
gold_monthly_sales.write.format("delta") \
    .mode("overwrite") \
    .save("/Volumes/retail_project/retail_db/retail_files/gold_monthly_sales/")

AVERAGE ORDER VALUE

In [0]:
from pyspark.sql.functions import countDistinct

gold_aov = (
    silver_sales
    .groupBy("customer_id")
    .agg(
        sum("total_amount").alias("total_spend"),
        countDistinct("transaction_id").alias("order_count")
    )
    .withColumn(
        "avg_order_value",
        col("total_spend") / col("order_count")
    )
)
display(gold_aov)


In [0]:
gold_aov.write.format("delta") \
    .mode("overwrite") \
    .save("/Volumes/retail_project/retail_db/retail_files/gold_avg_order_value/")
